# Notebook 01 — Data Understanding

**Purpose:** First look at the raw dataset. No changes made to the data here — only observation.

**Rule:** This notebook reads from `data/raw/`. It never writes to it.

---

## What is this notebook doing?

Before building any model or writing any cleaning code, we need to answer these questions:
1. How many schemes are in the dataset?
2. What columns do we have and what type of data is in each?
3. What does a real record look like?
4. How long are the text fields?
5. What unique values exist in categorical columns?

This is like reading a book's table of contents before diving in.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# pandas display settings — show full text in cells
pd.set_option('display.max_colwidth', 200)
pd.set_option('display.max_columns', 20)

print('Libraries loaded successfully.')

---
## 1. Load the raw data

We load the CSV from `data/raw/`. This is the only place we ever read raw data from.

In [ ]:
RAW_PATH = '../data/raw/updated_data.csv'

df = pd.read_csv(RAW_PATH)

print(f'Dataset loaded.')
print(f'Shape: {df.shape[0]} rows × {df.shape[1]} columns')

---
## 2. Column names and data types

**`dtype`** = data type of a column.  
- `object` = text (string) in pandas  
- `int64` = whole numbers  
- `float64` = decimal numbers  

All our scheme content columns should be `object` (text).

In [ ]:
print('Column names and dtypes:')
print('─' * 40)
for i, (col, dtype) in enumerate(df.dtypes.items()):
    print(f'  {i:>2}. {repr(col):<30}  {dtype}')

**Observation to note:** There is a column with an empty string name `''` between `schemeCategory` and `tags`. This is a known issue — it appears to be an empty column created during CSV export. We will confirm and drop it in Notebook 02.

---
## 3. First 3 rows — see what real data looks like

In [ ]:
# Transpose (T) makes it easier to read — columns become rows
df.head(3).T

---
## 4. Count of unique values per column

This tells us:
- High unique count → free text column (e.g., `scheme_name`, `details`)
- Low unique count → categorical column (e.g., `level` which is Central/State/District)

In [ ]:
print('Unique value counts per column:')
print('─' * 40)
for col in df.columns:
    n_unique = df[col].nunique()
    n_total  = len(df)
    pct      = n_unique / n_total * 100
    print(f'  {repr(col):<30}  {n_unique:>5} unique  ({pct:.1f}%)')

---
## 5. Categorical column deep-dive: `level`

`level` tells us whether the scheme is Central, State, or District level.  
This is an important filter for the recommendation engine.

In [ ]:
print('Value counts for `level`:')
print(df['level'].value_counts())
print()
print(f'Null count: {df["level"].isnull().sum()}')

---
## 6. Categorical column deep-dive: `schemeCategory`

`schemeCategory` is the sector/domain of each scheme (Education, Health, Agriculture, etc.).  
This is important for display and filtering.

In [ ]:
print('Top 20 scheme categories:')
print(df['schemeCategory'].value_counts().head(20))

---
## 7. Text field length analysis

TF-IDF works on text. We want to know if the text fields have enough content to be useful.

We measure length in **characters** here (word count analysis is in Notebook 04 — EDA).

In [ ]:
text_cols = ['details', 'benefits', 'eligibility', 'application', 'documents']

print('Text field length statistics (characters):')
print('─' * 70)

stats = []
for col in text_cols:
    lengths = df[col].dropna().str.len()
    stats.append({
        'column'  : col,
        'non_null': len(lengths),
        'min'     : int(lengths.min()),
        'median'  : int(lengths.median()),
        'max'     : int(lengths.max()),
        'mean'    : int(lengths.mean()),
    })

stats_df = pd.DataFrame(stats).set_index('column')
print(stats_df.to_string())

---
## 8. Look at the `tags` column

Tags are comma-separated keywords associated with each scheme.  
They will be part of the `combined_text` field we build for TF-IDF.

In [ ]:
print('Sample tags values (first 10 rows):')
for i, val in enumerate(df['tags'].dropna().head(10)):
    print(f'  [{i}] {val}')

---
## 9. Quick look at the unnamed column

The CSV header has 11 columns but one has an empty name. Let's check what it contains.

In [ ]:
# Find columns with empty or 'Unnamed' names
suspect_cols = [c for c in df.columns if str(c).strip() == '' or 'Unnamed' in str(c)]
print(f'Suspect columns: {suspect_cols}')

for col in suspect_cols:
    non_null = df[col].notna().sum()
    print(f'  Column {repr(col)}: {non_null} non-null values out of {len(df)}')
    if non_null > 0:
        print('  Sample non-null values:')
        print(df[col].dropna().head(5).to_list())
    else:
        print('  → Completely empty. Safe to drop.')

---
## 10. Summary — What we learned

After running all cells, fill in these observations:

| Question | Answer |
|---|---|
| Total records | _(run cell 2)_ |
| Confirmed columns | 11 (10 meaningful + 1 empty) |
| Level distribution | Central / State / District |
| Top categories | _(run cell 6)_ |
| Richest text field | _(run cell 7 — look at median length)_ |
| Empty column | Yes — safe to drop |

**Next step:** Notebook 02 — Data Quality will count missing values, find duplicates, and document everything that needs to be fixed.